In [1]:
!wget https://raw.githubusercontent.com/ye-kyaw-thu/sylbreak/master/python/sylbreak.py

--2025-12-06 14:07:25--  https://raw.githubusercontent.com/ye-kyaw-thu/sylbreak/master/python/sylbreak.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3668 (3.6K) [text/plain]
Saving to: ‘sylbreak.py’

sylbreak.py         100%[===================>]   3.58K  --.-KB/s    in 0s      

2025-12-06 14:07:25 (45.4 MB/s) - ‘sylbreak.py’ saved [3668/3668]



In [2]:
!pip install transformers==4.41.1 peft==0.11.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 63.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 63.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 99.4 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 87.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 34.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12

In [3]:
!pip install 'datasets[audio]==2.14.4' 'fsspec==2023.9.2'

INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 519.3/519.3 kB 8.3 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.4/173.4 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.3/115.3 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.4/135.4 kB 8.2 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.10.0
    Uninstalling fsspec-2025.10.0:
      Successfully uninstalled fsspec-2025.10.0
  Attempting uninstall: dill
    Found existing installation: dill 0.4.0
    Uninstalling dill-0.4.0:
      Successfully uninstalled dill-0.4.0
  Attempting uninstall: multiprocess
    Found existing installation: multiprocess 0.70.18
    Uninstalling multiprocess-0.70.18:
      Successfully uninstalled multiprocess-0.70.18
  Attempting uninstall: dataset

In [5]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
speech_data = load_dataset("LULab/mediTalk-mm-rdy", split='test')

Extracting data files:   0%|          | 0/2 [00:00<?, ?it/s]

Generating test split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

In [6]:
from sylbreak import break_syllables, create_break_pattern

def syllable_break(text):
  """Syllable break for burmese texts"""
  text = text
  separator = ' '
  break_pattern = create_break_pattern()

  segmented = break_syllables(text, break_pattern, separator)
  return segmented

In [7]:
def apply_syllable_break(text):
    text['prompt'] = syllable_break(text['prompt'])
    return text

dataset = speech_data.map(apply_syllable_break)
dataset

Map:   0%|          | 0/2920 [00:00<?, ? examples/s]

Dataset({
    features: ['speaker_id', 'prompt', 'audio'],
    num_rows: 2920
})

In [8]:
from datasets import Audio

dataset = speech_data.cast_column("audio", Audio(sampling_rate=16000))

In [9]:
from transformers import pipeline
import torch

MODEL_NAME = "YeBhoneLin10/Whisper-Medium-AUG-v5-retrain-latest"  # specify the model name
lang = "my"  # change to Thai langauge

device = 0 if torch.cuda.is_available() else "cpu"

pipe = pipeline(
    task="automatic-speech-recognition",
    model=MODEL_NAME,
    chunk_length_s=10,
    stride_length_s=2,
    device=device,
)
pipe.model.config.forced_decoder_ids = pipe.tokenizer.get_decoder_prompt_ids(
  language=lang,
  task="transcribe"
)

2025-12-06 14:10:33.690109: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765030234.088643      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765030234.212145      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


preprocessor_config.json:   0%|          | 0.00/339 [00:00<?, ?B/s]

In [13]:
#import torch
#from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
#from datasets import load_dataset


#device = "cuda:0" if torch.cuda.is_available() else "cpu"
#torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

#model_id = "YeBhoneLin10/Whisper-Medium-AUG-v5-retrain-latest"

#model = AutoModelForSpeechSeq2Seq.from_pretrained(
#    model_id, torch_dtype=torch_dtype, low_cpu_mem_usage=True
#)
#model.to(device)

#processor = AutoProcessor.from_pretrained(model_id)

# pipe = pipeline(
#    "automatic-speech-recognition",
#    model=model,
#    tokenizer=processor.tokenizer,
#    feature_extractor=processor.feature_extractor,
#    chunk_length_s=10,
#    stride_length_s=2,
#    batch_size=16,  # batch size for inference - set based on your device
#   torch_dtype=torch_dtype,
#    device=device)


#generate_kwargs = {
#    "max_new_tokens": 350,
#    "num_beams": 1,
#    "condition_on_prev_tokens": False,
    # "compression_ratio_threshold": 1.35,  # zlib compression ratio threshold (in token space)
#    "temperature": 0.0,
    # "logprob_threshold": -1.0,
    # "no_speech_threshold": 0.6
#}

#pipe.model.config.forced_decoder_ids = pipe.tokenizer.get_decoder_prompt_ids(
#  language="myanmar",
#  task="transcribe"
# )

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [10]:
from tqdm import tqdm
import pandas as pd

asr_outputs = []
correct_refs = []

for item in tqdm(dataset):

    audio_input = item["audio"]
    result = pipe(audio_input)

    asr_outputs.append(result["text"])
    correct_refs.append(item["prompt"])

df = pd.DataFrame({
    "ASR_Error": asr_outputs,
    "Correct": correct_refs
})

df.to_csv("FULL_Medium_AUGv4.tsv", sep="\t", index=False)

  0%|          | 0/2920 [00:00<?, ?it/s]/usr/local/lib/python3.11/dist-packages/transformers/generation/utils.py:1283: UserWarning: You have modified the pretrained model configuration to control generation. This is a deprecated strategy to control generation and will be removed soon, in a future version. Please use and modify the model generation configuration (see https://huggingface.co/docs/transformers/generation_strategies#default-text-generation-configuration )
  warnings.warn(
100%|██████████| 2920/2920 [3:37:08<00:00,  4.46s/it]  
